In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets scikit-learn --quiet

In [ ]:
#CHANGE THESE THREE LINES EACH RUN
MODEL_TYPE = "distilbert"   # tfidf | distilbert | bert | roberta
CORPUS     = "general"      # general | blockchain
ADAPT_FROM = None           # None | <transformer>_general






from pathlib import Path

DRIVE_ROOT   = Path("/content/drive/MyDrive/CryptoGuard")
DATA_ROOT    = DRIVE_ROOT / "data"
MODELS_ROOT  = DRIVE_ROOT / "models"
FIGURES_DIR  = DRIVE_ROOT / "outputs" / "figures"

MODELS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

HF_MODEL_MAP = {
    "distilbert": "distilbert-base-uncased",
    "bert":       "bert-base-uncased",
    "roberta":    "roberta-base",
}

if CORPUS == "general":
    TRAIN_PATH = DATA_ROOT / "general" / "train.csv"
    VAL_PATH   = DATA_ROOT / "general" / "val.csv"
    TEST_PATH  = DATA_ROOT / "general" / "test.csv"
else:
    TRAIN_PATH = DATA_ROOT / "blockchain" / "blockchain_train.csv"
    VAL_PATH   = DATA_ROOT / "blockchain" / "blockchain_val.csv"
    TEST_PATH  = DATA_ROOT / "blockchain" / "blockchain_test.csv"

RUN_NAME  = f"{MODEL_TYPE}_{CORPUS}"
SAVE_DIR  = MODELS_ROOT / RUN_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE  = 42
MAX_LEN       = 256
BATCH_SIZE    = 32
EPOCHS        = 3
WARMUP_STEPS  = 100
WEIGHT_DECAY  = 0.01
LEARNING_RATE = 1e-5 if (CORPUS == "blockchain" and ADAPT_FROM) else 2e-5

print("Run configuration")
print(f"  Run name     : {RUN_NAME}")
print(f"  Model type   : {MODEL_TYPE}")
print(f"  Corpus       : {CORPUS}")
print(f"  Adapt from   : {ADAPT_FROM}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs       : {EPOCHS}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Max length   : {MAX_LEN}")
print(f"  Save dir     : {SAVE_DIR}")
print(f"  Figures dir  : {FIGURES_DIR}")
print("─" * 55)

In [ ]:
import numpy as np
import pandas as pd
import json
import pickle
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay,
)

if MODEL_TYPE != "tfidf":
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        get_linear_schedule_with_warmup,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if MODEL_TYPE != "tfidf" else None
print(f"Device: {device}")

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_val   = pd.read_csv(VAL_PATH)
df_test  = pd.read_csv(TEST_PATH)

for df in [df_train, df_val, df_test]:
    df['label'] = df['label'].astype(int)

print(f"Train : {len(df_train):,}  |  Val : {len(df_val):,}  |  Test : {len(df_test):,}")
print(f"Train label dist : {df_train['label'].value_counts().sort_index().to_dict()}")
print(f"Val   label dist : {df_val['label'].value_counts().sort_index().to_dict()}")
print(f"Test  label dist : {df_test['label'].value_counts().sort_index().to_dict()}")

In [ ]:
def save_confusion_matrix(y_true, y_pred, run_name, figures_dir):
    """Saves a confusion matrix to figures_dir."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))

    model_label = run_name.replace('_', ' ').title()
    corpus_label = "General Email" if "general" in run_name else "Blockchain"

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Legitimate', 'Phishing'],
        yticklabels=['Legitimate', 'Phishing'],
        ax=ax,
        linewidths=0.5,
        linecolor='white',
    )
    ax.set_xlabel('Predicted label', fontsize=11)
    ax.set_ylabel('True label', fontsize=11)
    ax.set_title(f"{model_label}\n{corpus_label} test set", fontsize=12, fontweight='bold')

    plt.tight_layout()
    path = figures_dir / f"confusion_matrix_{run_name}.png"
    plt.savefig(str(path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {path}")
    plt.close()


def save_loss_curves(history, run_name, figures_dir):
    """Saves train/val loss and val F1 curves to figures_dir."""
    epochs_range = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Training curves — {run_name.replace('_', ' ').title()}",
                 fontsize=13, fontweight='bold')

    axes[0].plot(epochs_range, history['train_loss'], marker='o', label='Train loss')
    axes[0].plot(epochs_range, history['val_loss'],   marker='o', label='Val loss')
    axes[0].set_title('Loss per epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].spines[['top', 'right']].set_visible(False)

    axes[1].plot(epochs_range, history['val_f1'], marker='o', color='green', label='Val F1')
    axes[1].set_title('Validation F1 per epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1')
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()
    axes[1].spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    path = figures_dir / f"loss_curves_{run_name}.png"
    plt.savefig(str(path), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
if MODEL_TYPE == "tfidf":

    vectorizer = TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
    )

    X_train = vectorizer.fit_transform(df_train['text'])
    X_val   = vectorizer.transform(df_val['text'])
    X_test  = vectorizer.transform(df_test['text'])

    y_train = df_train['label'].values
    y_val   = df_val['label'].values
    y_test  = df_test['label'].values

    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
    clf.fit(X_train, y_train)

    val_preds = clf.predict(X_val)
    print(f"Val F1: {f1_score(y_val, val_preds):.4f}")

    test_preds = clf.predict(X_test)
    test_probs = clf.predict_proba(X_test)[:, 1]

    results = {
        "run_name":  RUN_NAME,
        "accuracy":  round(accuracy_score(y_test,  test_preds), 4),
        "precision": round(precision_score(y_test, test_preds), 4),
        "recall":    round(recall_score(y_test,    test_preds), 4),
        "f1":        round(f1_score(y_test,        test_preds), 4),
        "roc_auc":   round(roc_auc_score(y_test,   test_probs), 4),
    }

    print("\nTest Results")
    for k, v in results.items():
        print(f"  {k:12s}: {v}")

    #figures
    save_confusion_matrix(y_test, test_preds, RUN_NAME, FIGURES_DIR)

    #save artefacts
    with open(SAVE_DIR / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)
    with open(SAVE_DIR / "classifier.pkl", "wb") as f:
        pickle.dump(clf, f)
    with open(SAVE_DIR / "test_results.json", "w") as f:
        json.dump(results, f, indent=2)

    pd.DataFrame({
        "y_true": y_test,
        "y_pred": test_preds,
        "y_prob": test_probs,
    }).to_csv(SAVE_DIR / "test_predictions.csv", index=False)

    print("\nTF-IDF run complete")

In [ ]:
if MODEL_TYPE != "tfidf":

    HF_NAME = HF_MODEL_MAP[MODEL_TYPE]
    print(f"Loading tokeniser: {HF_NAME}")
    tokeniser = AutoTokenizer.from_pretrained(HF_NAME)

    class PhishingDataset(torch.utils.data.Dataset):
        def __init__(self, texts, labels, tokeniser, max_len):
            self.texts     = texts
            self.labels    = labels
            self.tokeniser = tokeniser
            self.max_len   = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tokeniser(
                str(self.texts[idx]),
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            return {
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label":          torch.tensor(self.labels[idx], dtype=torch.long),
            }

    train_ds = PhishingDataset(df_train['text'].tolist(), df_train['label'].tolist(), tokeniser, MAX_LEN)
    val_ds   = PhishingDataset(df_val['text'].tolist(),   df_val['label'].tolist(),   tokeniser, MAX_LEN)
    test_ds  = PhishingDataset(df_test['text'].tolist(),  df_test['label'].tolist(),  tokeniser, MAX_LEN)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}")

In [ ]:
if MODEL_TYPE != "tfidf":

    if ADAPT_FROM:
        checkpoint_path = str(MODELS_ROOT / ADAPT_FROM)
        print(f"Loading checkpoint for domain adaptation: {checkpoint_path}")
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path, num_labels=2)
    else:
        print(f"Loading pretrained weights: {HF_NAME}")
        model = AutoModelForSequenceClassification.from_pretrained(HF_NAME, num_labels=2)

    model.to(device)
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters — total: {total_params:,}  |  trainable: {trainable_params:,}")

In [ ]:
if MODEL_TYPE != "tfidf":

    total_steps = len(train_loader) * EPOCHS

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps,
    )

    print(f"Total training steps : {total_steps}")
    print(f"Warmup steps         : {WARMUP_STEPS}")
    print(f"Learning rate        : {LEARNING_RATE}")

In [ ]:
if MODEL_TYPE != "tfidf":

    history = {"train_loss": [], "val_loss": [], "val_f1": []}
    best_val_f1 = 0.0

    def evaluate_loader(loader):
        model.eval()
        total_loss, all_preds, all_labels = 0.0, [], []
        with torch.no_grad():
            for batch in loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["label"].to(device)
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                total_loss += outputs.loss.item()
                preds = outputs.logits.argmax(dim=-1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        return total_loss / len(loader), f1_score(all_labels, all_preds), all_preds, all_labels

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_train_loss = 0.0

        for step, batch in enumerate(train_loader, 1):
            optimizer.zero_grad()
            outputs = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                labels         = batch["label"].to(device),
            )
            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_train_loss += outputs.loss.item()

            if step % 50 == 0 or step == len(train_loader):
                print(f"  Epoch {epoch} | Step {step}/{len(train_loader)} | "
                      f"Batch loss: {outputs.loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_loader)
        val_loss, val_f1, _, _ = evaluate_loader(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        print(f"\nEpoch {epoch}/{EPOCHS} — "
              f"Train loss: {avg_train_loss:.4f}  "
              f"Val loss: {val_loss:.4f}  "
              f"Val F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model.save_pretrained(str(SAVE_DIR))
            tokeniser.save_pretrained(str(SAVE_DIR))
            print(f" New best val F1: {best_val_f1:.4f} — checkpoint saved")

    print(f"\nTraining complete. Best val F1: {best_val_f1:.4f}")

In [ ]:
if MODEL_TYPE != "tfidf":
    save_loss_curves(history, RUN_NAME, FIGURES_DIR)

In [ ]:
if MODEL_TYPE != "tfidf":

    #loading best checkpoint
    best_model = AutoModelForSequenceClassification.from_pretrained(str(SAVE_DIR))
    best_model.to(device)
    best_model.eval()

    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
            preds   = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch["label"].numpy())
            all_probs.extend(probs)

    y_test  = np.array(all_labels)
    y_pred  = np.array(all_preds)
    y_probs = np.array(all_probs)

    results = {
        "run_name":          RUN_NAME,
        "accuracy":          round(accuracy_score(y_test,  y_pred),  4),
        "precision":         round(precision_score(y_test, y_pred),  4),
        "recall":            round(recall_score(y_test,    y_pred),  4),
        "f1":                round(f1_score(y_test,        y_pred),  4),
        "roc_auc":           round(roc_auc_score(y_test,   y_probs), 4),
        "training_history":  history,
    }

    print("\nTest Results")
    for k, v in results.items():
        if k != "training_history":
            print(f"  {k:12s}: {v}")

    #figures
    save_confusion_matrix(y_test, y_pred, RUN_NAME, FIGURES_DIR)

    #save artefacts
    with open(SAVE_DIR / "test_results.json", "w") as f:
        json.dump(results, f, indent=2)

    pd.DataFrame({
        "y_true": y_test,
        "y_pred": y_pred,
        "y_prob": y_probs,
    }).to_csv(SAVE_DIR / "test_predictions.csv", index=False)

In [ ]:
if MODEL_TYPE != "tfidf":

    with open(SAVE_DIR / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print("training summary")
    for epoch, (tl, vl, vf) in enumerate(
        zip(history['train_loss'], history['val_loss'], history['val_f1']), 1
    ):
        print(f"  Epoch {epoch}: train_loss={tl:.4f}  val_loss={vl:.4f}  val_f1={vf:.4f}")

print(f"RUN COMPLETE: {RUN_NAME}")